# 제품별 배치 프롬프트 생성 (옵션 C, v3: product_info (1).json 사용)

In [ ]:
# =============================
# 0) CONFIG
# =============================
from pathlib import Path

PERSONA_JSONL = Path("/mnt/data/persona_attributes_weighted.jsonl")
PRODUCT_JSON  = Path("/mnt/data/product_info (1).json")
OUT_JSONL     = Path("/mnt/data/prompts_C.jsonl")
OUT_PREVIEW   = Path("/mnt/data/prompts_C_preview.json")

BATCH_SIZE = 10  # 5~30 권장
print("CONFIG loaded.")

In [ ]:
# =============================
# 1) Load data
# =============================
import json

# Personas
personas = []
with open(PERSONA_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            personas.append(json.loads(line))

# Products from JSON
products_raw = json.loads(PRODUCT_JSON.read_text(encoding="utf-8"))

def as_list(val):
    if val is None: return []
    parts = [p.strip() for p in str(val).replace(";", ",").split(",")]
    return [p for p in parts if p]

def price_text(v):
    try:
        iv = int(v)
        return f"{iv:,}원"
    except Exception:
        return str(v) if v is not None else None

def to_product_dict(r):
    cat = r.get("category") or {}
    features = as_list(r.get("feature"))
    targeted = as_list(r.get("targeted_consumer"))
    return {
        "product_id": r.get("product_id") or r.get("id") or "",
        "product_name": r.get("product_name"),
        "category": cat.get("level_1"),
        "subcategory": " > ".join([c for c in [cat.get("level_2"), cat.get("level_3")] if c]),
        "features": features,
        "targeted_consumer": targeted,
        "launch_ym": r.get("launch_ym"),
        "price": price_text(r.get("price")),
        "ad_model": r.get("ad_model"),
        "advertise": r.get("advertise"),
    }

def product_block(p):
    lines = [
        f"- product_id: {p.get('product_id','')}",
        f"- 제품명: {p.get('product_name','')}"
    ]
    if p.get("category"): lines.append(f"- 카테고리: {p['category']}")
    if p.get("subcategory"): lines.append(f"- 서브카테고리: {p['subcategory']}")
    if p.get("features"): lines.append(f"- 주요 특징: {', '.join(p['features'])}")
    if p.get("targeted_consumer"): lines.append(f"- 타깃: {', '.join(p['targeted_consumer'])}")
    if p.get("price"): lines.append(f"- 기준 가격대: {p['price']}")
    if p.get("advertise"): lines.append(f"- 광고/프로모션: {p['advertise']}")
    return "\n".join(lines)

products = [to_product_dict(r) for r in products_raw]
print("Loaded:", len(personas), "personas /", len(products), "products")

In [ ]:
# =============================
# 2) Helpers
# =============================
from typing import List, Dict, Any

def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def format_attributes_for_prompt(attrs: Dict[str, Any]) -> str:
    lines = []
    for k, vw in attrs.items():
        v = vw.get("value", None)
        w = vw.get("weight", 0.0)
        v_str = "None" if v is None else str(v)
        lines.append(f"- {k}: {v_str} (w={w:.3f})")
    return "\n".join(lines[:40])

def persona_to_prompt_block(p: Dict[str, Any]) -> str:
    meta = p.get("meta", {}) or {}
    cluster = meta.get("cluster", "")
    label = meta.get("label", "")
    desc = meta.get("desc", meta.get("Description",""))
    cluster_block = f"- cluster: {cluster}\n- label: {label}\n- desc: {desc}" if (cluster or label or desc) else "- cluster: N/A"
    return f"""
[페르소나]
- id: {p.get('persona_key','')}
- 속성(가중치 합=1):
{format_attributes_for_prompt(p.get('attributes', {}))}
- 클러스터 컨텍스트:
{cluster_block}
""".strip()

def build_batch_prompt(product: Dict[str, Any], persona_batch: List[Dict[str, Any]]) -> str:
    return f"""
[역할]
당신은 한국 소비자 데이터 분석가이자 마케팅 전문가입니다.
아래의 "제품 정보"와 "페르소나 목록"을 바탕으로, 각 페르소나마다
해당 제품의 구매자 페르소나를 **싱글 턴**으로 완결된 JSON 객체로 생성하세요.
각 페르소나는 서로 독립적이며, 서로의 정보에 영향을 주지 마세요.

[제품 정보]
{product_block(product)}

[페르소나 목록]
{ "\n\n".join(persona_to_prompt_block(p) for p in persona_batch) }

[규칙]
- '클러스터 컨텍스트'는 페르소나의 배경 지침으로만 사용합니다. 속성 가중치(합=1)와 충돌 시 '속성 가중치'를 우선합니다.
- 2024-07 ~ 2025-06 월별로 구매확률(prob 0~1)과 예상수량(qty 정수)을 제시합니다.
- 추석/설, 광고/프로모션/계절성을 반영합니다.
- **반드시 아래 JSON 스키마(JSON 배열)를 출력**하고, 불필요한 설명 문장은 출력하지 마세요.

[출력 스키마(JSON 배열)]
[
  {{
    "persona_id": "p_{{product_id}}_{{persona_key}}",
    "product_id": "{{product_id}}",
    "segment_ref": "{{persona_key}}",
    "attributes": {{ "{{속성명}}": {{"value": "<값>", "weight": <0~1> }}, "...": "..." }},
    "purchase_pattern": {{
      "avg_purchase_prob": <0~1>,
      "avg_purchase_qty": <int>,
      "seasonality": {{"추석": "+x%", "설": "+y%"}},
      "promotion_effect": "광고/프로모션 노출 시 +z%"
    }},
    "forecast_12mo": {{
      "2024-07": {{"prob": <0~1>, "qty": <int>}},
      "...": {{}}, 
      "2025-06": {{"prob": <0~1>, "qty": <int>}}
    }}
  }},
  ...
]
""".strip()

In [ ]:
# =============================
# 3) Build & save
# =============================
import json

records = []
for prod in products:
    for batch in chunked(personas, BATCH_SIZE):
        rec = {
            "product": prod,
            "personas": [{"persona_key": p.get("persona_key")} for p in batch],
            "prompt": build_batch_prompt(prod, batch)
        }
        records.append(rec)

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

from pathlib import Path
Path(OUT_PREVIEW).write_text(json.dumps(records[:1], ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", OUT_JSONL, "size=", Path(OUT_JSONL).stat().st_size, "bytes")
records[:1]